# Lab 7.2 — Retrieval Evaluation

Measure baseline retrieval quality on 25 hand-written questions and identify failure modes that Lab 7.3 will try to fix.

**Dual metrics**: every question carries both a metadata target (`company, fiscal_year, section_id`) **and** keyword anchors (`must_contain_any`). We report Hit@k and MRR under both.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT.name != 'financial-report-analyst' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

Project root: C:\repos\financial-report-analyst


## 1. Inspect the question set

In [2]:
import pandas as pd
from financial_analyst.evaluate import load_questions

questions = load_questions()
print(f'{len(questions)} questions loaded')
from collections import Counter
cats = Counter(q.category for q in questions)
pd.DataFrame(cats.items(), columns=['category', 'n']).sort_values('n', ascending=False)

c:\repos\financial-report-analyst\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


25 questions loaded


,category,n
0,single,10
1,comparison_year,6
2,comparison_company,5
3,risk,3
4,strategy,1


In [3]:
# Show the first 5 questions in detail
for q in questions[:5]:
    print(f'{q.id} [{q.category}]: {q.question}')
    for e in q.expected:
        print(f'   -> expect: {e.company} FY{e.fiscal_year} {e.section_id}  must_contain_any={e.must_contain_any}')
    print()

q01 [single]: How did NVIDIA's data center revenue change from fiscal 2023 to fiscal 2024?
   -> expect: nvidia FY2024 item_7  must_contain_any=['Data Center', 'data center']

q02 [single]: What were Apple's iPhone revenue results in fiscal year 2024?
   -> expect: apple FY2024 item_7  must_contain_any=['iPhone']

q03 [single]: How did Microsoft's Intelligent Cloud segment perform in fiscal year 2024?
   -> expect: microsoft FY2024 item_7  must_contain_any=['Intelligent Cloud', 'Azure']

q04 [single]: What did NVIDIA say about its Hopper GPU platform in the FY2024 10-K?
   -> expect: nvidia FY2024 item_7  must_contain_any=['Hopper', 'H100']

q05 [single]: How does Microsoft govern cybersecurity risk according to its FY2024 disclosure?
   -> expect: microsoft FY2024 item_1c  must_contain_any=['cybersecurity', 'governance', 'Board']



## 2. Run the baseline evaluation

In [4]:
from financial_analyst.evaluate import evaluate, write_eval_json, write_eval_markdown, write_failure_log
from financial_analyst.index import IndexConfig, load_index

index = load_index(IndexConfig())
summary = evaluate(index, top_k=5, use_filters=True, config_name='baseline')

metrics_df = pd.DataFrame([
    {'metric': 'Metadata Hit@5', 'value': summary.metadata_hit_rate},
    {'metric': 'Strict   Hit@5', 'value': summary.strict_hit_rate},
    {'metric': 'Metadata MRR',   'value': summary.metadata_mrr},
    {'metric': 'Strict   MRR',   'value': summary.strict_mrr},
])
metrics_df

Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from C:\repos\financial-report-analyst\storage\c512_o50_bge-small-en-v1.5_5c38ec7c_9e02b0f25518\index_store.json.


,metric,value
0,Metadata Hit@5,0.840000
1,Strict Hit@5,0.760000
2,Metadata MRR,0.633333
3,Strict MRR,0.603333


## 3. Per-question results

In [5]:
per_q = pd.DataFrame([{
    'id': r.question_id,
    'category': r.category,
    'metadata_rank': r.metadata_first_rank,
    'strict_rank': r.strict_first_rank,
    'top1_citation': r.hits[0].citation if r.hits else None,
} for r in summary.per_question])
per_q

,id,category,metadata_rank,strict_rank,top1_citation
0,q01,single,1.0,1.0,NVIDIA FY2024 — Item 7 (Management's Discussio...
1,q02,single,1.0,1.0,APPLE FY2024 — Item 7 (Management's Discussion...
2,q03,single,2.0,3.0,MICROSOFT FY2024 — Item 8 (Financial Statement...
3,q04,single,1.0,1.0,NVIDIA FY2024 — Item 7 (Management's Discussio...
4,q05,single,1.0,1.0,MICROSOFT FY2024 — Item 1C (Cybersecurity)
5,q06,single,1.0,1.0,APPLE FY2024 — Item 1A (Risk Factors)
6,q07,single,1.0,1.0,NVIDIA FY2025 — Item 1A (Risk Factors)
7,q08,single,NaN,NaN,MICROSOFT FY2024 — Item 1 (Business)
8,q09,single,1.0,1.0,APPLE FY2024 — Item 7 (Management's Discussion...
9,q10,single,1.0,1.0,NVIDIA FY2024 — Item 1A (Risk Factors)


## 4. Metrics by category

Comparison questions (cross-year and cross-company) tend to score worse — they have more ground-truth slots to hit and similar-vocabulary distractors across years.

In [6]:
import statistics
rows = []
for cat in sorted({r.category for r in summary.per_question}):
    items = [r for r in summary.per_question if r.category == cat]
    rows.append({
        'category': cat,
        'n': len(items),
        'metadata_hit@5': sum(1 for r in items if r.metadata_first_rank) / len(items),
        'strict_hit@5':   sum(1 for r in items if r.strict_first_rank)   / len(items),
        'strict_mrr':     statistics.fmean(r.strict_mrr for r in items),
    })
pd.DataFrame(rows)

,category,n,metadata_hit@5,strict_hit@5,strict_mrr
0,comparison_company,5,0.6,0.600000,0.183333
1,comparison_year,6,1.0,0.833333,0.722222
2,risk,3,1.0,0.666667,0.500000
3,single,10,0.9,0.900000,0.833333
4,strategy,1,0.0,0.000000,0.000000


## 5. Failure log — three worst-scoring questions

Generated to `results/failures_baseline.md`.

In [7]:
fail_path = write_failure_log(summary, k_worst=3)
print(open(fail_path, encoding='utf-8').read())

# Retrieval Failure Log — `baseline`

The 3 worst-performing questions by strict MRR (then metadata MRR).

## q08 — `single`

**Question:** What does Microsoft say about its AI strategy and Copilot products in fiscal 2024?

- Metadata first-rank: **not in top-k**
- Strict first-rank:   **not in top-k**
- Inferred filters: companies=['microsoft'], years=[2024]

### Top retrievals

- **[1] (miss)** MICROSOFT FY2024 — Item 1 (Business)
  > These PCs use on-device AI for enhanced performance and features. Copilot is an AI assistant that helps users navigate the web, answer questions, and create content. Microsoft Edge is our fast and sec...
- **[2] (miss)** MICROSOFT FY2024 — Item 1 (Business)
  > Copilot Studio allows customers to customize Copilot for Microsoft 365 or build their own Copilot. Microsoft Power Platform helps domain experts drive productivity gains with low-code/no-code tools, r...
- **[3] (miss)** MICROSOFT FY2024 — Item 1 (Business)
  > Item 1 The Ambitions That Drive Us 

## 6. Persist eval artifacts

In [8]:
print('JSON:    ', write_eval_json(summary))
print('Markdown:', write_eval_markdown(summary))

JSON:     C:\repos\financial-report-analyst\results\eval_baseline.json
Markdown: C:\repos\financial-report-analyst\results\eval_baseline.md
